# Copernicus Marine Delivery – Python demo

In [1]:
import os
from pathlib import Path

for line in Path("../.env").read_text().splitlines():
    if line.startswith("export ") and "=" in line:
        key, value = line[len("export "):].split("=", 1)
        os.environ.setdefault(key, value.strip('"'))

# yaml file paths are relative to the CWD
os.chdir("delivery_multiple_operations")
os.environ.pop("PYTHON_LOG_DEBUG", None)

'false'

In [2]:
PRODUCT_ID = "GLOBAL_ANALYSISFORECAST_PHY_001_024"
DATASET_ID = "cmems_mod_glo_phy-cur_anfc_0.083deg_PT6H-i_202406"

## 1. Build the delivery

A delivery is a list of operations that run in order. Delete 4 files, then upload 16 (4 of them re-use a deleted name).

In [3]:
from copernicusmarine_delivery import Delete, Delivery, Upload, delivery_status, list_deliveries

delivery = Delivery()

# 1st operation: delete files already published in the dataset (S3 key suffixes)
delivery.add(
    Delete(
        files=[
            "2026/05/glo12_rg_6h-i_20260527-06h_3D-uovo_nwct_R20260603.nc",
            "2026/05/glo12_rg_6h-i_20260527-12h_3D-uovo_nwct_R20260603.nc",
            "2026/05/glo12_rg_6h-i_20260527-18h_3D-uovo_nwct_R20260603.nc",
            "2026/05/glo12_rg_6h-i_20260528-00h_3D-uovo_nwct_R20260603.nc",
        ]
    )
)

# 2nd operation: upload local files (all .nc files below 2026/)
delivery.add(Upload(files=[str(f) for f in sorted(Path("2026").rglob("*.nc"))]))

for op in delivery.operations:
    print(type(op).__name__, len(op.files), "files")

Delete 4 files
Upload 4 files


## 2. Dry run

Validates product, dataset and files. Nothing is uploaded.

In [4]:
response = delivery.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID, dry_run=True)
print("Fatal error:", response.fatal_error)

2026-11-03 11:11:43 - copernicusmarine-delivery - INFO - [DRY RUN]: Starting to upload 4 files ...
2026-11-03 11:11:43 - copernicusmarine-delivery - INFO - [DRY RUN]: Finished uploading 4 files for a total size of 29.00 MB.


Fatal error: None


## 3. Submit

In [5]:
from copernicusmarine_delivery import ResponseUpload, ResponseDelete


response = delivery.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID)

if response.fatal_error:
    print(f"Delivery failed: {response.fatal_error}")
else:
    print(response)

2026-11-03 11:12:15 - copernicusmarine-delivery - INFO - Starting to upload 4 files ...
2026-11-03 11:12:16 - copernicusmarine-delivery - INFO - Uploaded [1/4] file glo12_rg_6h-i_20260527-18h_3D-uovo_nwct_R20260603.nc
2026-11-03 11:12:18 - copernicusmarine-delivery - INFO - Uploaded [2/4] file glo12_rg_6h-i_20260527-06h_3D-uovo_nwct_R20260603.nc
2026-11-03 11:12:18 - copernicusmarine-delivery - INFO - Uploaded [3/4] file glo12_rg_6h-i_20260528-00h_3D-uovo_nwct_R20260603.nc
2026-11-03 11:12:19 - copernicusmarine-delivery - INFO - Uploaded [4/4] file glo12_rg_6h-i_20260527-12h_3D-uovo_nwct_R20260603.nc
2026-11-03 11:12:19 - copernicusmarine-delivery - INFO - Finished uploading 4 files for a total size of 29.00 MB.



Delivery summary:

	delivery_id: 20261103T111214-GLOBAL_ANALYSISFORECAST_PHY_001_024-3681
	pushing_entity_id: GLO-MERCATOR-TOULOUSE-FR
	product_id: GLOBAL_ANALYSISFORECAST_PHY_001_024
	dataset_id: cmems_mod_glo_phy-cur_anfc_0.083deg_PT6H-i_202406

	Total: 4 files uploaded, 4 files to delete.



## 4. Status

Reuses the `delivery_id` from the submit above.

In [6]:
import time

def wait_for_status(delivery_id: str):
    """Poll until the delivery is completed or failed, then show errors."""
    for _ in range(10):
        status = delivery_status(delivery_id)
        print(status.status.value)
        if status.status.value in ("completed", "failed"):
            break
        time.sleep(10)
    print(f"Delivery error: {status.error_message}" if status.error_message else "No errors in delivery!")
    for op in status.operations:
        for key_suffix, error in op.get_errored_file_names():
            print(f"  {op.operation.value} {key_suffix}: {error}")
    return status

DELIVERY_ID = response.delivery_id
status = wait_for_status(DELIVERY_ID)

completed
No errors in delivery!


## 5. List deliveries

In [7]:
for d in list_deliveries(timeframe="h"):
    print(d.delivery_id, d.status.value)

20261103T111214-GLOBAL_ANALYSISFORECAST_PHY_001_024-3681 completed
20261103T110800-GLOBAL_ANALYSISFORECAST_PHY_001_024-5875 completed
20261103T110530-GLOBAL_ANALYSISFORECAST_PHY_001_024-4372 completed
20261103T110026-GLOBAL_ANALYSISFORECAST_PHY_001_024-7351 completed


## Error cases

### A. Toolbox detects the error

Two of the files do not exist locally. The toolbox validates all files **before** uploading anything, so nothing is uploaded and no delivery is created.

In [8]:
from copernicusmarine_delivery import InvalidFilesError

delivery_a = Delivery()
delivery_a.add(
    Upload(
        files=[
            "2026/05/glo12_rg_6h-i_20260527-06h_3D-uovo_nwct_R20260603.nc",
            "2026/05/glo12_rg_6h-i_20260527-12h_3D-uovo_nwct_R20260603.nc",
            "2026/05/glo12_rg_6h-i_20260527-18h_3D-uovo_nwct_R20260603.nc",
            "2026/05/this_file_does_not_exist_locally_1.nc",
            "2026/06/this_file_does_not_exist_locally_2.nc",
        ]
    )
)

try:
    delivery_a.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID)
except InvalidFilesError as error:
    print("Toolbox error:", error.invalid_files)

2026-11-03 11:14:13 - copernicusmarine-delivery - ERROR - Invalid file: 2026/05/this_file_does_not_exist_locally_1.nc: File path does not exist.
2026-11-03 11:14:13 - copernicusmarine-delivery - ERROR - Invalid file: 2026/06/this_file_does_not_exist_locally_2.nc: File path does not exist.


Toolbox error: [2026/05/this_file_does_not_exist_locally_1.nc: File path does not exist., 2026/06/this_file_does_not_exist_locally_2.nc: File path does not exist.]


### B. OPDV detects the error

The toolbox cannot know what is in the dataset, so it accepts a delete of a file that does not exist. OPDV rejects it afterwards.

In [9]:
delivery_b = Delivery()
delivery_b.add(Delete(files=["2026/05/glo12_rg_6h-i_20200101-00h_3D-uovo_nwct_DOES_NOT_EXIST.nc"]))
delivery_b.add(Upload(files=["2026/05/glo12_rg_6h-i_20260527-06h_3D-uovo_nwct_R20260603.nc"]))

response_b = delivery_b.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID)
print(f"Delivery ID: {response_b.delivery_id}")
status_b = wait_for_status(response_b.delivery_id)

2026-11-03 11:14:43 - copernicusmarine-delivery - INFO - Starting to upload 1 files ...
2026-11-03 11:14:45 - copernicusmarine-delivery - INFO - Uploaded [1/1] file glo12_rg_6h-i_20260527-06h_3D-uovo_nwct_R20260603.nc
2026-11-03 11:14:45 - copernicusmarine-delivery - INFO - Finished uploading 1 files for a total size of 6.00 MB.


Delivery ID: 20261103T111443-GLOBAL_ANALYSISFORECAST_PHY_001_024-9106
completed
Delivery error: Some operations failed. Last job being: VALIDATE
  delete 2026/05/glo12_rg_6h-i_20200101-00h_3D-uovo_nwct_DOES_NOT_EXIST.nc: File not found at S3 path: native/GLOBAL_ANALYSISFORECAST_PHY_001_024/cmems_mod_glo_phy-cur_anfc_0.083deg_PT6H-i_202406/2026/05/glo12_rg_6h-i_20200101-00h_3D-uovo_nwct_DOES_NOT_EXIST.nc


### C. Delivery OK, error in publication

Toolbox and OPDV accept the delivery. The API is hardcoded to fail the publication of the file named `*_PUBERROR.nc`.

In [10]:
import os

from copernicusmarine_delivery import Delivery, Upload

os.chdir("../delivery_publication_error")

delivery_c = Delivery()
delivery_c.add(
    Upload(
        files=[
            "2026/06/glo12_rg_6h-i_20260603-00h_3D-uovo_nwct_R20260604.nc",
            "2026/06/glo12_rg_6h-i_20260603-06h_3D-uovo_nwct_R20260604.nc",
            "2026/06/glo12_rg_6h-i_20260603-12h_3D-uovo_nwct_PUBERROR.nc",
        ]
    )
)

response_c = delivery_c.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID)
print(f"Delivery ID: {response_c.delivery_id}")
status_c = wait_for_status(response_c.delivery_id)

2026-11-03 11:15:36 - copernicusmarine-delivery - INFO - Starting to upload 3 files ...
2026-11-03 11:15:37 - copernicusmarine-delivery - INFO - Uploaded [1/3] file glo12_rg_6h-i_20260603-12h_3D-uovo_nwct_PUBERROR.nc
2026-11-03 11:15:37 - copernicusmarine-delivery - INFO - Uploaded [2/3] file glo12_rg_6h-i_20260603-00h_3D-uovo_nwct_R20260604.nc
2026-11-03 11:15:37 - copernicusmarine-delivery - INFO - Uploaded [3/3] file glo12_rg_6h-i_20260603-06h_3D-uovo_nwct_R20260604.nc
2026-11-03 11:15:37 - copernicusmarine-delivery - INFO - Finished uploading 3 files for a total size of 3.00 MB.


Delivery ID: 20261103T111536-GLOBAL_ANALYSISFORECAST_PHY_001_024-2827
validated
validated
validated
completed
Delivery error: Some operations failed. Last job being: UPLOAD
  upload 2026/06/glo12_rg_6h-i_20260603-12h_3D-uovo_nwct_PUBERROR.nc: 2026/06/glo12_rg_6h-i_20260603-12h_3D-uovo_nwct_PUBERROR.nc - The upload step failed. Please contact User Support.
